In [55]:
import os, sys
import numpy as np
import torch
import espaloma as esp
import qcportal as ptl
from collections import Counter
from openff.toolkit.topology import Molecule
from openff.qcsubmit.results import BasicResultCollection
from simtk import unit
from simtk.unit import Quantity
from matplotlib import pyplot as plt

In [56]:
def get_graph(mol, energy, grad):
    offmol = Molecule.from_qcschema(mol, allow_undefined_stereo=True)   # convert to OpenFF Molecule object
    offmol.compute_partial_charges_am1bcc()   # https://docs.openforcefield.org/projects/toolkit/en/0.9.2/api/generated/openff.toolkit.topology.Molecule.html
    charges = offmol.partial_charges.value_in_unit(esp.units.CHARGE_UNIT)
    g = esp.Graph(offmol)
    
    # energy is already hartree
    g.nodes["g"].data["u_ref"] = torch.tensor(
        [
            Quantity(
                energy,
                esp.units.HARTREE_PER_PARTICLE,
            ).value_in_unit(esp.units.ENERGY_UNIT)
        ],
        dtype=torch.get_default_dtype(),
    )[None, :]

    g.nodes["n1"].data["xyz"] = torch.tensor(
        np.stack(
            [
                Quantity(
                    mol.geometry,
                    unit.bohr,
                ).value_in_unit(esp.units.DISTANCE_UNIT)
            ],
            axis=1,
        ),
        requires_grad=True,
        dtype=torch.get_default_dtype(),
    )

    g.nodes["n1"].data["u_ref_prime"] = torch.stack(
        [
            torch.tensor(
                Quantity(
                    grad,
                    esp.units.HARTREE_PER_PARTICLE / unit.bohr,
                ).value_in_unit(esp.units.FORCE_UNIT),
                dtype=torch.get_default_dtype(),
            )
        ],
        dim=1,
    )
    
    g.nodes['n1'].data['q_ref'] = c = torch.tensor(charges, dtype=torch.get_default_dtype(),).unsqueeze(-1)
    
    return g

In [57]:
#ds = esp.data.dataset.GraphDataset.load("mydata")

In [58]:
collection_type = "Dataset"
name = "RNA Single Point Dataset v1.0"

client = ptl.FractalClient()
collection = client.get_collection(collection_type, name)

In [59]:
recs_wb97m = collection.get_records(method='wb97m-d3bj', basis='def2-tzvppd', program='psi4', keywords='wb97m-d3bj/def2-tzvppd')

In [60]:
#recs_wb97m

In [61]:
gs = []

for i in range(len(recs_wb97m)):
    if recs_wb97m.iloc[i].record.status == 'COMPLETE':
        print("#{}: {}".format(i, recs_wb97m.iloc[i].name))
        
        mol = client.query_molecules(recs_wb97m.iloc[i].record.molecule)[0]
        energy = recs_wb97m.iloc[i].record.properties.scf_total_energy
        grad = recs_wb97m.iloc[i].record.return_result
        
        g = get_graph(mol, energy, grad)
        gs.append(g)
        #break

Warning (not error because allow_undefined_stereo=True): OEMol has unspecified stereochemistry. oemol.GetTitle(): 
Problematic atoms are:
Atom atomic num: 15, name: , idx: 29, aromatic: False, chiral: True with bonds:
bond order: 1, chiral: False to atom atomic num: 8, name: , idx: 28, aromatic: False, chiral: False
bond order: 1, chiral: False to atom atomic num: 8, name: , idx: 30, aromatic: False, chiral: False
bond order: 2, chiral: False to atom atomic num: 8, name: , idx: 31, aromatic: False, chiral: False
bond order: 1, chiral: False to atom atomic num: 8, name: , idx: 32, aromatic: False, chiral: False
Atom atomic num: 15, name: , idx: 63, aromatic: False, chiral: True with bonds:
bond order: 1, chiral: False to atom atomic num: 8, name: , idx: 62, aromatic: False, chiral: False
bond order: 1, chiral: False to atom atomic num: 8, name: , idx: 64, aromatic: False, chiral: False
bond order: 2, chiral: False to atom atomic num: 8, name: , idx: 65, aromatic: False, chiral: False
bo

#506: Nc1ccn([C@@H]2O[C@H](CO)[C@@H](OP(=O)([O-])OC[C@H]3O[C@@H](n4cnc5c(=O)[nH]c(N)nc54)[C@H](O)[C@@H]3OP(=O)([O-])OC[C@H]3O[C@@H](n4cnc5c(=O)[nH]c(N)nc54)[C@H](O)[C@@H]3O)[C@H]2O)c(=O)n1-14


Warning (not error because allow_undefined_stereo=True): OEMol has unspecified stereochemistry. oemol.GetTitle(): 
Problematic atoms are:
Atom atomic num: 15, name: , idx: 29, aromatic: False, chiral: True with bonds:
bond order: 1, chiral: False to atom atomic num: 8, name: , idx: 10, aromatic: False, chiral: False
bond order: 1, chiral: False to atom atomic num: 8, name: , idx: 30, aromatic: False, chiral: False
bond order: 2, chiral: False to atom atomic num: 8, name: , idx: 31, aromatic: False, chiral: False
bond order: 1, chiral: False to atom atomic num: 8, name: , idx: 32, aromatic: False, chiral: False
Atom atomic num: 15, name: , idx: 63, aromatic: False, chiral: True with bonds:
bond order: 1, chiral: False to atom atomic num: 8, name: , idx: 41, aromatic: False, chiral: False
bond order: 1, chiral: False to atom atomic num: 8, name: , idx: 64, aromatic: False, chiral: False
bond order: 2, chiral: False to atom atomic num: 8, name: , idx: 65, aromatic: False, chiral: False
bo

#528: Nc1ccn([C@@H]2O[C@H](CO)[C@@H](OP(=O)([O-])OC[C@H]3O[C@@H](n4cnc5c(=O)[nH]c(N)nc54)[C@H](O)[C@@H]3OP(=O)([O-])OC[C@H]3O[C@@H](n4cnc5c(=O)[nH]c(N)nc54)[C@H](O)[C@@H]3O)[C@H]2O)c(=O)n1-34


Warning (not error because allow_undefined_stereo=True): OEMol has unspecified stereochemistry. oemol.GetTitle(): 
Problematic atoms are:
Atom atomic num: 15, name: , idx: 29, aromatic: False, chiral: True with bonds:
bond order: 1, chiral: False to atom atomic num: 8, name: , idx: 10, aromatic: False, chiral: False
bond order: 1, chiral: False to atom atomic num: 8, name: , idx: 30, aromatic: False, chiral: False
bond order: 2, chiral: False to atom atomic num: 8, name: , idx: 31, aromatic: False, chiral: False
bond order: 1, chiral: False to atom atomic num: 8, name: , idx: 32, aromatic: False, chiral: False
Atom atomic num: 15, name: , idx: 63, aromatic: False, chiral: True with bonds:
bond order: 1, chiral: False to atom atomic num: 8, name: , idx: 41, aromatic: False, chiral: False
bond order: 1, chiral: False to atom atomic num: 8, name: , idx: 64, aromatic: False, chiral: False
bond order: 2, chiral: False to atom atomic num: 8, name: , idx: 65, aromatic: False, chiral: False
bo

#536: Nc1ccn([C@@H]2O[C@H](CO)[C@@H](OP(=O)([O-])OC[C@H]3O[C@@H](n4cnc5c(=O)[nH]c(N)nc54)[C@H](O)[C@@H]3OP(=O)([O-])OC[C@H]3O[C@@H](n4cnc5c(=O)[nH]c(N)nc54)[C@H](O)[C@@H]3O)[C@H]2O)c(=O)n1-41


KeyboardInterrupt: 

In [ ]:
ds = esp.data.dataset.GraphDataset(gs)

In [ ]:
len(ds)

In [ ]:
dir = "mydata"
if os.path.exists(dir):
    shutil.rmtree(dir)
ds.save(dir)

In [ ]:
ds.shuffle(seed=2666)
#ds_tr, ds_vl, ds_te = ds.split([8, 1, 1])
ds_tr, ds_vl, ds_te = ds.split([1, 1, 8])

In [ ]:
print("train:    ", len(ds_tr))
print("validate: ", len(ds_vl))
print("test:     ", len(ds_te))

In [ ]:
ds_tr_loader = ds_tr.view(batch_size=128, shuffle=True)

In [ ]:
representation = esp.nn.Sequential(
    layer=esp.nn.layers.dgl_legacy.gn("SAGEConv"),   # use SAGEConv implementation in DGL
    config=[128, "relu", 128, "relu", 128, "relu"],  # 3 layers, 128 units, ReLU activation
)

In [ ]:
readout = esp.nn.readout.janossy.JanossyPooling(
    in_features=128, config=[128, "relu", 128, "relu", 128, "relu"],
    out_features={                  # define modular MM parameters Espaloma will assign
        1: {"e": 1, "s": 1},        # atom hardness and electronegativity
        2: {"log_coefficients": 2}, # bond linear combination, enforce positive
        3: {"log_coefficients": 2}, # angle linear combination, enforce positive
        4: {"k": 6},                # torsion barrier heights (can be positive or negative)
    },
)

espaloma_model = torch.nn.Sequential(
                 representation, readout, esp.nn.readout.janossy.ExpCoefficients(),
                 esp.mm.geometry.GeometryInGraph(),
                 esp.mm.energy.EnergyInGraph(),
                 esp.nn.readout.charge_equilibrium.ChargeEquilibrium(),
)

In [ ]:
if torch.cuda.is_available():
    espaloma_model = espaloma_model.cuda()

In [ ]:
loss_fn = [
    esp.metrics.GraphMetric(
        base_metric=torch.nn.MSELoss(), # use mean-squared error loss
        between=["u", "u_ref"],         # between predicted and QM energies
        level="g",                      # compare on graph level
    ),
    esp.metrics.GraphMetric(
        base_metric=torch.nn.MSELoss(), # use mean-squared error loss
        between=["q", "q_hat"],         # between predicted and reference charges
        level="n1",                     # compare on node level 
    )
]

In [ ]:
epochs = 5
optimizer = torch.optim.Adam(espaloma_model.parameters(), 1e-4)

In [ ]:
for epoch in range(epochs):
    #for g in ds_tr:
    for g in ds_tr_loader:
        optimizer.zero_grad()
        if torch.cuda.is_available():
            #g.heterograph = g.heterograph.to("cuda:0")
            g.to("cuda:0")
        #g = espaloma_model(g.heterograph)
        g = espaloma_model(g)
        
        #loss = loss_fn(g)
        loss = loss_fn[0](g)
        loss += loss_fn[1](g)
        
        loss.backward()
        optimizer.step()
        torch.save(espaloma_model.state_dict(), "%s.pt" % epoch)

torch.save(espaloma_model.state_dict(), "espaloma_model.pt")

In [ ]:
inspect_metric = loss_fn
#inspect_metric = esp.metrics.center(torch.nn.MSELoss(), reduction="mean") # use mean-squared error loss

In [ ]:
loss_tr = []
loss_vl = []
loss_te = []

In [ ]:
with torch.no_grad():
    for idx_epoch in range(epochs):
        espaloma_model.load_state_dict(
            torch.load("%s.pt" % idx_epoch)
        )

        # training set performance
        u = []
        u_ref = []
        for g in ds_tr:
            if torch.cuda.is_available():
                g.heterograph = g.heterograph.to("cuda:0")
            espaloma_model(g.heterograph)
            u.append(g.nodes['g'].data['u'])
            #u_ref.append(g.nodes['g'])
            u_ref.append(g.nodes['g'].data['u_ref'])
        #u = torch.cat(u, dim=0)
        #u_ref = torch.cat(u_ref, dim=0)
        u = torch.cat(u, dim=1)
        u_ref = torch.cat(u_ref, dim=1)
        loss_tr.append(inspect_metric(u, u_ref))


        # validation set performance
        u = []
        u_ref = []
        for g in ds_vl:
            if torch.cuda.is_available():
                g.heterograph = g.heterograph.to("cuda:0")
            espaloma_model(g.heterograph)
            u.append(g.nodes['g'].data['u'])
            #u_ref.append(g.nodes['g'])
            u_ref.append(g.nodes['g'].data['u_ref'])
        #u = torch.cat(u, dim=0)
        #u_ref = torch.cat(u_ref, dim=0)
        u = torch.cat(u, dim=1)
        u_ref = torch.cat(u_ref, dim=1)
        loss_vl.append(inspect_metric(u, u_ref))
        
        
        # test set performance
        u = []
        u_ref = []
        for g in ds_te:
            if torch.cuda.is_available():
                g.heterograph = g.heterograph.to("cuda:0")
            espaloma_model(g.heterograph)
            u.append(g.nodes['g'].data['u'])
            #u_ref.append(g.nodes['g'])
            u_ref.append(g.nodes['g'].data['u_ref'])
        #u = torch.cat(u, dim=0)
        #u_ref = torch.cat(u_ref, dim=0)
        u = torch.cat(u, dim=1)
        u_ref = torch.cat(u_ref, dim=1)
        loss_te.append(inspect_metric(u, u_ref))

In [ ]:
# hartee to kcal/mol
loss_tr = np.array(loss_tr) * 627.5
loss_vl = np.array(loss_vl) * 627.5
loss_te = np.array(loss_te) * 627.5

In [ ]:
plt.plot(loss_tr, label="train")
plt.plot(loss_vl, label="valid")
plt.plot(loss_te, label="test")
plt.yscale("log")
plt.legend()
plt.show()
plt.savefig("plot_loss.png")